# Narrowing down variant sites causative to stem juiciness phenotype
Assumed to be found on SBC4, 10, and 23 exclusively, based on the association with the phenotype.

In [1]:
# open vcf file
import numpy as np
import cyvcf2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gzip, re

sns.set_theme(style="whitegrid")

SAMPLE   = "SBC4, 10, and 23"
VCF_PATH = "../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz"
GFF_PATH = "../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz"

## Overall steps
1. Prepare dataframe
2. Scoring variant sites based on its impact & risk score
3. Convert variant sites into gene using 4 different approaches

These steps is executed by the following script

In [4]:
!cd .. && ./docker/run.sh python3 workflow/scripts/genomics_scoring.py -i results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz -o /home/daffa/Work/2026/thesis/results/gene_score/ -g resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz

[genomics_scoring] Parsing VCF: results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz
[genomics_scoring]   205,996 (variant, gene) rows
[genomics_scoring] Loading gene lengths: resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz
[genomics_scoring]   32,458 genes with length
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv  (12,917 genes)
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv  (12,917 genes)
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv  (12,917 genes)
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv  (12,917 genes)
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.scores_summary.png


## Analysis

In [6]:
df_scored_gene_max      = pd.read_csv("/home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv", sep="\t")
df_scored_gene_sum      = pd.read_csv("/home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv", sep="\t") 
df_scored_gene_max_norm = pd.read_csv("/home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv", sep="\t")
df_scored_gene_sum_norm = pd.read_csv("/home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv", sep="\t")

In [7]:
df_scored_gene_max

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score
0,LOC8076751,LOC8076751,protein_coding,NC_012874.2,0.875
1,LOC8085125,LOC8085125,protein_coding,NC_012878.2,0.875
2,LOC8062456,LOC8062456,protein_coding,NC_012872.2,0.875
3,LOC110437519,LOC110437519,protein_coding,NC_012877.2,0.875
4,LOC8079394,LOC8079394,protein_coding,NC_012873.2,0.875
...,...,...,...,...,...
12912,LOC8057980,LOC8057980,protein_coding,NC_012876.2,0.125
12913,LOC8057980-LOC110437112,LOC8057980-LOC110437112,NaN,NC_012876.2,0.125
12914,LOC8057981-LOC8057982,LOC8057981-LOC8057982,NaN,NC_012876.2,0.125
12915,LOC8057982,LOC8057982,protein_coding,NC_012876.2,0.125


In [8]:
df_scored_gene_max_norm

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score_raw,gene_length,score
0,LOC110435944,LOC110435944,protein_coding,NC_012874.2,0.875,309.0,2.831715
1,LOC110437609,LOC110437609,protein_coding,NC_012877.2,0.875,342.0,2.558480
2,LOC110432712,LOC110432712,protein_coding,NC_012871.2,0.875,372.0,2.352151
3,LOC110434814,LOC110434814,protein_coding,NC_012873.2,0.875,396.0,2.209596
4,LOC8081850,LOC8081850,protein_coding,NC_012870.2,0.875,414.0,2.113527
...,...,...,...,...,...,...,...
12912,LOC8057977-LOC8057978,LOC8057977-LOC8057978,NaN,NC_012876.2,0.125,NaN,NaN
12913,LOC8057978-TRNAV-CAC_8,LOC8057978-TRNAV-CAC_8,NaN,NC_012876.2,0.125,NaN,NaN
12914,LOC8057979-LOC8057980,LOC8057979-LOC8057980,NaN,NC_012876.2,0.125,NaN,NaN
12915,LOC8057980-LOC110437112,LOC8057980-LOC110437112,NaN,NC_012876.2,0.125,NaN,NaN


In [9]:
df_scored_gene_sum

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score,n_rows
0,LOC110429702-LOC110437569,LOC110429702-LOC110437569,NaN,NC_012877.2,190.875,1527
1,LOC8066633-LOC8066635,LOC8066633-LOC8066635,NaN,NC_012878.2,185.000,1480
2,LOC8086468-LOC8069451,LOC8086468-LOC8069451,NaN,NC_012875.2,157.250,1258
3,LOC110436224-LOC110436226,LOC110436224-LOC110436226,NaN,NC_012875.2,153.500,1228
4,LOC110436439-LOC110436440,LOC110436439-LOC110436440,NaN,NC_012875.2,142.125,1137
...,...,...,...,...,...,...
12912,LOC8072243,LOC8072243,protein_coding,NC_012871.2,0.125,1
12913,LOC8072248,LOC8072248,protein_coding,NC_012871.2,0.125,1
12914,LOC8072248-LOC8072249,LOC8072248-LOC8072249,NaN,NC_012871.2,0.125,1
12915,LOC8072249,LOC8072249,protein_coding,NC_012871.2,0.125,1


In [10]:
df_scored_gene_sum_norm

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score_raw,n_rows,gene_length,score
0,LOC110435944,LOC110435944,protein_coding,NC_012874.2,26.6675,180,309.0,86.302589
1,TRNAM-CAU,TRNAM-CAU,protein_coding,NC_012870.2,3.2500,26,72.0,45.138889
2,LOC8083563,LOC8083563,protein_coding,NC_012876.2,31.4400,212,757.0,41.532365
3,LOC8081850,LOC8081850,protein_coding,NC_012870.2,14.4350,80,414.0,34.867150
4,LOC8071174,LOC8071174,protein_coding,NC_012876.2,22.2500,174,669.0,33.258595
...,...,...,...,...,...,...,...,...
12912,LOC8072409-LOC8085802,LOC8072409-LOC8085802,NaN,NC_012872.2,0.1250,1,NaN,NaN
12913,LOC8072448-LOC8072449,LOC8072448-LOC8072449,NaN,NC_012873.2,0.1250,1,NaN,NaN
12914,LOC110432664-LOC110432665,LOC110432664-LOC110432665,NaN,NC_012871.2,0.1250,1,NaN,NaN
12915,LOC8072180-LOC8072181,LOC8072180-LOC8072181,NaN,NC_012876.2,0.1250,1,NaN,NaN
